# BertDiffused — Colab Training Pipeline

End-to-end notebook for training BertMoEDiffusion on Google Colab:
1. **Setup** — Install deps, mount Drive, clone repo
2. **ETL** — Download LM1B, clean, deduplicate, tokenize, shard to Parquet
3. **Train** — LoRA + MoE fine-tuning with MLflow tracking
4. **Monitor** — Live loss curves, eval BPD, parameter breakdown
5. **Save** — Only LoRA adapters + final merged model to Drive (space-efficient)

> **Drive space strategy**: We save only LoRA adapters (~2.5 MB) during training 
> and one final merged model at the end. Intermediate full checkpoints are kept 
> only on the Colab VM (ephemeral) — not synced to Drive.

## 0. GPU Check & Environment

In [ ]:
import torch

assert torch.cuda.is_available(), "GPU not available — go to Runtime > Change runtime type > GPU"
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

## 1. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Persistent output directory on Drive (small footprint)
DRIVE_OUTPUT = "/content/drive/MyDrive/BertDiffused"
!mkdir -p {DRIVE_OUTPUT}/lora_adapters
!mkdir -p {DRIVE_OUTPUT}/final_model
!mkdir -p {DRIVE_OUTPUT}/mlruns
print(f"Drive output dir: {DRIVE_OUTPUT}")

In [ ]:
import os
os.chdir('/content')

# Clone the repo (or pull latest if already cloned)
REPO_URL = "https://github.com/stephenlee/BertDiffused.git"  # UPDATE with your repo URL
if not os.path.exists('/content/BertDiffused'):
    !git clone {REPO_URL}
else:
    !cd /content/BertDiffused && git pull

os.chdir('/content/BertDiffused')
print(f"Working dir: {os.getcwd()}")

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import mlflow
import mlflow.pytorch
import numpy as np
import yaml
import logging
import math
import time as time_module
from pathlib import Path
from IPython.display import display, HTML, clear_output
import matplotlib.pyplot as plt

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

print(f"MLflow version: {mlflow.__version__}")

## 3. Load Configuration

In [ ]:
with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

# ── Colab overrides ────────────────────────────────────────────────────────
# Reduce batch size for Colab GPU memory
cfg["training"]["batch_size"] = 32
cfg["training"]["gradient_accumulation_steps"] = 16  # effective batch = 512
cfg["training"]["max_steps"] = 50000                 # shorter run for Colab
cfg["training"]["save_steps"] = 10000
cfg["training"]["eval_steps"] = 5000
cfg["training"]["log_steps"] = 50
cfg["training"]["output_dir"] = "/content/checkpoints"  # ephemeral (Colab VM)

# ETL: limit samples for Colab (set to None for full dataset)
cfg["etl"]["max_samples"] = 500000   # 500K samples — adjust based on time budget
cfg["etl"]["output_dir"] = "/content/data_processed"
cfg["etl"]["num_proc"] = 2

# MLflow: store on Drive for persistence
cfg["mlflow"]["tracking_uri"] = f"{DRIVE_OUTPUT}/mlruns"

# LoRA: enabled by default (parameter-efficient)
cfg["model"]["lora"]["enabled"] = True

print("Config loaded with Colab overrides.")
print(f"  Batch: {cfg['training']['batch_size']} x {cfg['training']['gradient_accumulation_steps']} = {cfg['training']['batch_size'] * cfg['training']['gradient_accumulation_steps']}")
print(f"  Max steps: {cfg['training']['max_steps']:,}")
print(f"  LoRA: rank={cfg['model']['lora']['rank']}, alpha={cfg['model']['lora']['alpha']}")
print(f"  ETL samples: {cfg['etl']['max_samples']:,}")

## 4. ETL Pipeline — Download, Clean, Tokenize, Shard

Runs the full Extract → Transform → Load pipeline.  
Output: Parquet shards in `/content/data_processed/{train,test}/`

In [ ]:
from transformers import AutoTokenizer
from data.etl import DataETL

tokenizer = AutoTokenizer.from_pretrained(cfg["model"]["backbone"])
mask_token_id = tokenizer.mask_token_id

# Setup MLflow for ETL tracking
mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
mlflow.set_experiment(cfg["mlflow"]["experiment_name"])

etl = DataETL(cfg, tokenizer, output_dir=cfg["etl"]["output_dir"])

with mlflow.start_run(run_name="colab-etl"):
    mlflow.set_tag("pipeline_stage", "etl")
    mlflow.set_tag("runtime", "colab")

    print("\n" + "="*60)
    print("ETL: Processing TRAIN split")
    print("="*60)
    etl.run(split="train")

    print("\n" + "="*60)
    print("ETL: Processing TEST split")
    print("="*60)
    # Use smaller test set
    cfg_test = cfg.copy()
    etl_test = DataETL(cfg_test, tokenizer, output_dir=cfg["etl"]["output_dir"])
    etl_test.run(split="test")

print("\nETL Stats:")
for k, v in etl.stats.items():
    print(f"  {k}: {v:,.2f}" if isinstance(v, float) else f"  {k}: {v:,}")

In [ ]:
# Verify ETL output
import glob

train_shards = sorted(glob.glob(f"{cfg['etl']['output_dir']}/train/shard-*.parquet"))
test_shards = sorted(glob.glob(f"{cfg['etl']['output_dir']}/test/shard-*.parquet"))

print(f"Train shards: {len(train_shards)}")
print(f"Test shards:  {len(test_shards)}")

# Check disk usage
!du -sh {cfg['etl']['output_dir']}/train/ {cfg['etl']['output_dir']}/test/ 2>/dev/null || echo "Shards not found"

## 5. Build Model (LoRA + MoE)

In [ ]:
from model import BertMoEDiffusion, LogLinearNoiseSchedule

moe_cfg = cfg["model"]["moe"]
lora_cfg = cfg["model"]["lora"]

model = BertMoEDiffusion(
    bert_model_name=cfg["model"]["backbone"],
    moe_layers=moe_cfg["moe_layers"],
    num_experts=moe_cfg["num_experts"],
    num_experts_per_token=moe_cfg["num_experts_per_token"],
    expert_hidden_multiplier=moe_cfg["expert_hidden_multiplier"],
    router_jitter=moe_cfg["router_jitter"],
    router_z_loss_coef=moe_cfg["router_z_loss_coef"],
    router_aux_loss_coef=moe_cfg["router_aux_loss_coef"],
    time_embed_dim=cfg["model"]["time_embed_dim"],
    use_time_conditioning=cfg["model"]["use_time_conditioning"],
    dropout=cfg["model"]["dropout"],
    lora_enabled=lora_cfg["enabled"],
    lora_rank=lora_cfg["rank"],
    lora_alpha=lora_cfg["alpha"],
    lora_dropout=lora_cfg["dropout"],
    lora_target_modules=lora_cfg["target_modules"],
)
model.set_mask_token_id(mask_token_id)

device = torch.device("cuda")
model.to(device)

# Parameter summary
ps = model.trainable_parameters_summary()
print("\n" + "="*50)
print("MODEL PARAMETER SUMMARY")
print("="*50)
print(f"Total params:     {ps['total']:>12,}")
print(f"Trainable params: {ps['trainable']:>12,}  ({ps['trainable_pct']:.1f}%)")
print(f"Frozen params:    {ps['frozen']:>12,}")
print(f"├── LoRA params:  {ps['lora']:>12,}")
print(f"└── MoE params:   {ps['moe']:>12,}")
print(f"\nGPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")

## 6. Prepare Data Loaders

In [ ]:
from torch.utils.data import DataLoader
from data.etl import ProcessedParquetDataset
from data.lm1b_dataset import LM1BDataset

train_etl_dir = Path(cfg["etl"]["output_dir"]) / "train"
eval_etl_dir = Path(cfg["etl"]["output_dir"]) / "test"

if train_etl_dir.exists() and any(train_etl_dir.glob("shard-*.parquet")):
    print("Using preprocessed ETL Parquet shards")
    train_dataset = ProcessedParquetDataset(train_etl_dir)
    eval_dataset = ProcessedParquetDataset(eval_etl_dir)
else:
    print("No ETL shards found — falling back to raw on-the-fly tokenization")
    train_dataset = LM1BDataset(split="train", tokenizer=tokenizer, max_seq_len=cfg["model"]["max_seq_len"])
    eval_dataset = LM1BDataset(split="test", tokenizer=tokenizer, max_seq_len=cfg["model"]["max_seq_len"])

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg["training"]["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
eval_loader = DataLoader(
    eval_dataset,
    batch_size=cfg["evaluation"]["eval_batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

print(f"Train: {len(train_dataset):,} samples, {len(train_loader):,} batches")
print(f"Eval:  {len(eval_dataset):,} samples")

## 7. Training Loop with MLflow Tracking

**Storage strategy** (Drive-friendly):
- Intermediate checkpoints → Colab VM only (ephemeral, overwritten each save)
- LoRA adapters → Drive (~2.5 MB per save)
- Final merged model → Drive (one copy)
- MLflow tracking DB → Drive (persistent across sessions)

In [ ]:
import torch.nn.functional as F
from transformers import get_linear_schedule_with_warmup

# ── Loss function (from train.py) ──────────────────────────────────────────
def compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps=1e-4):
    B, L, V = logits.shape
    weights = 1.0 / t.clamp(min=time_eps)
    is_masked = (z_t == mask_token_id)
    ce = F.cross_entropy(
        logits.reshape(B * L, V), input_ids.reshape(B * L), reduction='none'
    ).reshape(B, L)
    ce = ce * is_masked.float()
    loss_per_seq = ce.sum(-1)
    return (weights * loss_per_seq).mean()

# ── Eval BPD (from train.py) ───────────────────────────────────────────────
@torch.no_grad()
def evaluate_bpd(model, dataloader, noise_schedule, mask_token_id, device,
                 num_eval_steps=50, time_eps=1e-4):
    model.eval()
    total_loss, total_tokens, n_batches = 0.0, 0, 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        B, L = input_ids.shape
        batch_loss = 0.0
        for _ in range(num_eval_steps):
            t = noise_schedule.sample_t(B, device, low_discrepancy=True)
            z_t = noise_schedule.noise_sequence(input_ids, t, mask_token_id)
            logits = model(z_t, t, attention_mask=attention_mask)
            loss = compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps)
            batch_loss += loss.item()
        total_loss += batch_loss / num_eval_steps
        total_tokens += B * L
        n_batches += 1
        if n_batches >= 20:
            break
    avg_nll = total_loss / n_batches
    return avg_nll / (math.log(2) * (total_tokens / (n_batches * input_ids.shape[0])))

print("Loss and eval functions ready.")

In [ ]:
# ── Setup optimizer, scheduler, scaler ─────────────────────────────────────
noise_schedule = LogLinearNoiseSchedule()

trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_params,
    lr=cfg["training"]["learning_rate"],
    betas=(cfg["training"]["adam_beta1"], cfg["training"]["adam_beta2"]),
    eps=cfg["training"]["adam_epsilon"],
    weight_decay=cfg["training"]["weight_decay"],
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=cfg["training"]["warmup_steps"],
    num_training_steps=cfg["training"]["max_steps"],
)
scaler = torch.cuda.amp.GradScaler(enabled=cfg["training"]["fp16"])

print(f"Optimizer: AdamW, LR={cfg['training']['learning_rate']}, {len(trainable_params)} param groups")
print(f"Scheduler: linear warmup ({cfg['training']['warmup_steps']} steps) + decay")

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────

# Tracking lists for live plots
history = {"step": [], "elbo_loss": [], "moe_aux": [], "lr": [], "bpd": [], "bpd_step": []}

accum_steps = cfg["training"]["gradient_accumulation_steps"]
max_steps = cfg["training"]["max_steps"]
log_steps = cfg["training"]["log_steps"]
eval_steps = cfg["training"]["eval_steps"]
save_steps = cfg["training"]["save_steps"]
time_eps = cfg["diffusion"]["time_eps"]
use_fp16 = cfg["training"]["fp16"]
output_dir = Path(cfg["training"]["output_dir"])
output_dir.mkdir(parents=True, exist_ok=True)

# MLflow run
mlflow.set_tracking_uri(cfg["mlflow"]["tracking_uri"])
mlflow.set_experiment(cfg["mlflow"]["experiment_name"])
mlflow_run = mlflow.start_run(run_name="colab-training")

# Flatten and log config
def flatten_cfg(cfg, prefix=""):
    flat = {}
    for k, v in cfg.items():
        key = f"{prefix}.{k}" if prefix else k
        if isinstance(v, dict):
            flat.update(flatten_cfg(v, key))
        else:
            flat[key] = v
    return flat

mlflow.log_params(flatten_cfg(cfg))
mlflow.set_tags({"runtime": "colab", "lora_enabled": str(lora_cfg["enabled"]),
                 "gpu": torch.cuda.get_device_name(0)})
mlflow.log_metrics({f"model/{k}": v for k, v in ps.items()})

# ── Loop ───────────────────────────────────────────────────────────────────
model.train()
optimizer.zero_grad()
running_loss = 0.0
running_moe_loss = 0.0
global_step = 0
train_iter = iter(train_loader)
t_start = time_module.time()

print(f"Training started — {max_steps:,} steps, device={device}")
print(f"Checkpoints (VM): {output_dir}")
print(f"LoRA adapters (Drive): {DRIVE_OUTPUT}/lora_adapters/")
print("-" * 60)

try:
    while global_step < max_steps:
        try:
            batch = next(train_iter)
        except StopIteration:
            train_iter = iter(train_loader)
            batch = next(train_iter)

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch.get("attention_mask")
        if attention_mask is not None:
            attention_mask = attention_mask.to(device)
        B, L = input_ids.shape

        t = noise_schedule.sample_t(B, device, low_discrepancy=True)
        z_t = noise_schedule.noise_sequence(input_ids, t, mask_token_id)

        with torch.cuda.amp.autocast(enabled=use_fp16):
            logits = model(z_t, t, attention_mask=attention_mask)
            elbo_loss = compute_mdlm_loss(logits, input_ids, z_t, t, mask_token_id, time_eps)
            moe_aux = model.moe_aux_loss
            total_loss = (elbo_loss + moe_aux) / accum_steps

        scaler.scale(total_loss).backward()
        running_loss += elbo_loss.item()
        running_moe_loss += moe_aux.item() if isinstance(moe_aux, torch.Tensor) else moe_aux

        if (global_step + 1) % accum_steps == 0 or global_step == max_steps - 1:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["training"]["max_grad_norm"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        global_step += 1

        # ── Logging ────────────────────────────────────────────────────────
        if global_step % log_steps == 0:
            avg_loss = running_loss / log_steps
            avg_moe = running_moe_loss / log_steps
            lr = scheduler.get_last_lr()[0]
            elapsed = time_module.time() - t_start
            steps_per_sec = global_step / elapsed
            eta_min = (max_steps - global_step) / steps_per_sec / 60

            history["step"].append(global_step)
            history["elbo_loss"].append(avg_loss)
            history["moe_aux"].append(avg_moe)
            history["lr"].append(lr)

            mlflow.log_metrics(
                {"train/elbo_loss": avg_loss, "train/moe_aux_loss": avg_moe,
                 "train/learning_rate": lr, "train/steps_per_sec": steps_per_sec},
                step=global_step,
            )

            print(
                f"Step {global_step:6d}/{max_steps} | ELBO {avg_loss:.4f} | "
                f"MoE {avg_moe:.4f} | LR {lr:.2e} | "
                f"{steps_per_sec:.1f} steps/s | ETA {eta_min:.0f}m"
            )
            running_loss = 0.0
            running_moe_loss = 0.0

        # ── Evaluation ─────────────────────────────────────────────────────
        if global_step % eval_steps == 0:
            bpd = evaluate_bpd(model, eval_loader, noise_schedule, mask_token_id, device, time_eps=time_eps)
            history["bpd"].append(bpd)
            history["bpd_step"].append(global_step)
            mlflow.log_metric("eval/bpd", bpd, step=global_step)
            print(f"  >>> Eval BPD: {bpd:.4f}")
            model.train()

        # ── Checkpoint (VM only — overwrite to save space) ─────────────────
        if global_step % save_steps == 0:
            # Overwrite single checkpoint on VM (ephemeral)
            ckpt_path = output_dir / "latest_checkpoint.pt"
            torch.save(
                {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "global_step": global_step,
                 "config": cfg},
                ckpt_path,
            )
            print(f"  Checkpoint saved (VM): {ckpt_path}")

            # Save LoRA adapters to Drive (tiny, ~2.5 MB)
            lora_state = {k: v.cpu() for k, v in model.state_dict().items()
                          if "lora_A" in k or "lora_B" in k}
            lora_drive_path = f"{DRIVE_OUTPUT}/lora_adapters/lora_step_{global_step}.pt"
            torch.save({"lora_state_dict": lora_state, "global_step": global_step,
                         "config": cfg}, lora_drive_path)
            print(f"  LoRA adapters saved (Drive): {lora_drive_path}")

except KeyboardInterrupt:
    print("\nTraining interrupted by user.")

print(f"\nTraining finished at step {global_step}.")

## 8. Training Monitoring — Live Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("BertDiffused Training Monitor", fontsize=14, fontweight="bold")

# ELBO Loss
axes[0, 0].plot(history["step"], history["elbo_loss"], "b-", linewidth=0.8)
axes[0, 0].set_title("ELBO Loss")
axes[0, 0].set_xlabel("Step")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)

# MoE Auxiliary Loss
axes[0, 1].plot(history["step"], history["moe_aux"], "r-", linewidth=0.8)
axes[0, 1].set_title("MoE Auxiliary Loss")
axes[0, 1].set_xlabel("Step")
axes[0, 1].set_ylabel("Aux Loss")
axes[0, 1].grid(True, alpha=0.3)

# Learning Rate
axes[1, 0].plot(history["step"], history["lr"], "g-", linewidth=0.8)
axes[1, 0].set_title("Learning Rate")
axes[1, 0].set_xlabel("Step")
axes[1, 0].set_ylabel("LR")
axes[1, 0].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
axes[1, 0].grid(True, alpha=0.3)

# Eval BPD
if history["bpd"]:
    axes[1, 1].plot(history["bpd_step"], history["bpd"], "mo-", linewidth=1.2, markersize=6)
    axes[1, 1].set_title("Eval BPD (bits-per-dim)")
    axes[1, 1].set_xlabel("Step")
    axes[1, 1].set_ylabel("BPD")
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, "No eval data yet", ha='center', va='center', fontsize=12)
    axes[1, 1].set_title("Eval BPD")

plt.tight_layout()
plt.savefig(f"{DRIVE_OUTPUT}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Plot saved to {DRIVE_OUTPUT}/training_curves.png")

## 9. Save Final Model to Drive

Merge LoRA weights into base BERT → save a single inference-ready model.  
This is the **only full-size checkpoint** stored on Drive.

In [ ]:
# Save final LoRA adapters (before merge)
lora_final = {k: v.cpu() for k, v in model.state_dict().items()
              if "lora_A" in k or "lora_B" in k}
lora_final_path = f"{DRIVE_OUTPUT}/lora_adapters/lora_final.pt"
torch.save({"lora_state_dict": lora_final, "global_step": global_step, "config": cfg},
           lora_final_path)
print(f"Final LoRA adapters: {lora_final_path}")
mlflow.log_artifact(lora_final_path, artifact_path="lora_adapters")

# Merge LoRA into base weights (zero inference overhead)
model.merge_lora()
print("LoRA weights merged into base model.")

# Save merged model to Drive
final_path = f"{DRIVE_OUTPUT}/final_model/final_model.pt"
torch.save({"model": model.state_dict(), "config": cfg, "global_step": global_step},
           final_path)
print(f"Final merged model: {final_path}")

# Log to MLflow Model Registry
mlflow.log_artifact(final_path, artifact_path="final_model")
mlflow.pytorch.log_model(
    pytorch_model=model,
    artifact_path="model",
    registered_model_name=cfg["mlflow"]["registered_model_name"],
    pip_requirements=["torch>=2.2.0", "transformers>=4.40.0", "mlflow>=2.14.0"],
)
print(f"Model registered in MLflow as '{cfg['mlflow']['registered_model_name']}'")

# End MLflow run
mlflow.end_run()
print("MLflow run ended.")

## 10. Drive Storage Summary

In [ ]:
print("Files saved to Google Drive:")
print("=" * 50)
!du -sh {DRIVE_OUTPUT}/*
print()
print("LoRA adapter snapshots:")
!ls -lh {DRIVE_OUTPUT}/lora_adapters/
print()
print("Final model:")
!ls -lh {DRIVE_OUTPUT}/final_model/

## 11. Resume Training (if Colab disconnects)

Run this cell to resume from the latest checkpoint on the VM,  
or reload LoRA adapters from Drive if the VM was recycled.

In [ ]:
# ── Resume helpers ─────────────────────────────────────────────────────────

def resume_from_vm_checkpoint(model, optimizer, scheduler, ckpt_path="/content/checkpoints/latest_checkpoint.pt"):
    """Resume from full checkpoint on Colab VM (if session is still alive)."""
    if not os.path.exists(ckpt_path):
        print(f"No VM checkpoint found at {ckpt_path}")
        return 0
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    step = ckpt["global_step"]
    print(f"Resumed from VM checkpoint at step {step:,}")
    return step

def resume_from_drive_lora(model, drive_output=DRIVE_OUTPUT):
    """Reload the latest LoRA adapters from Drive (if VM was recycled)."""
    import glob
    lora_files = sorted(glob.glob(f"{drive_output}/lora_adapters/lora_step_*.pt"))
    if not lora_files:
        print("No LoRA adapters found on Drive.")
        return 0
    latest = lora_files[-1]
    ckpt = torch.load(latest, map_location=device)
    # Load only LoRA params into model
    model_state = model.state_dict()
    model_state.update(ckpt["lora_state_dict"])
    model.load_state_dict(model_state)
    step = ckpt.get("global_step", 0)
    print(f"Loaded LoRA adapters from Drive: {latest} (step {step:,})")
    print("NOTE: Optimizer/scheduler state is lost — training resumes with fresh optimizer.")
    return step

# Uncomment one of these to resume:
# global_step = resume_from_vm_checkpoint(model, optimizer, scheduler)
# global_step = resume_from_drive_lora(model)